# 04 — Train univariate AE & upload to KQL `models`

Local pipeline (runs on the repo `.venv`, no Lakehouse needed):

1. Pull the last `lookback` of a `(machine_id, sensor_id)` series from `raw_telemetry`.
2. Build non-overlapping windows of `WINDOW_SIZE` samples.
3. Train a small **LSTM autoencoder** in PyTorch.
4. Export a `ScoreWrapper` to **ONNX** (output = MSE per window → a single scalar score).
5. Upload the model (base64) into the `models` table as a new `version`.
6. Smoke-test by calling `score_univariate_onnx(...)` directly in KQL.

Prereqs: simulator has been running at least once (so `raw_telemetry` has data) and the `python` plugin is enabled on the Eventhouse (warm-up complete).

## 0. Install runtime dependencies (Fabric-only)

In Microsoft Fabric the Spark Python runtime ships with a curated package set
that does not include azure-kusto-data. The cell below installs the SDK on
first run; on a local .venv it is a no-op. For repeated runs in production,
prefer attaching a Fabric Environment with these libraries pre-published.


In [ ]:
# Fabric runtime does not preinstall azure-kusto-data; install it on first run.
# %pip restarts the Python kernel, so this MUST be the first executed cell.
# On a local .venv these packages are already present and pip is a no-op.
%pip install -q azure-kusto-data azure-identity python-dotenv


## 1. Config

In [4]:
MACHINE      = "M-001"
SENSOR       = "temperature_motor"
WINDOW_SIZE  = 64
LOOKBACK     = "30m"        # KQL timespan literal (we just started ingesting)
EPOCHS       = 30
MODEL_NAME   = f"univariate_ae__{SENSOR}"

## 2. Auth + Kusto client (dual mode: local `.venv` or Fabric notebook)

Same notebook runs both:
- **Local** (`.venv`): loads `.env`, uses the cached device-code credential from `tools/_fabric_auth.py`.
- **Fabric notebook**: detects `notebookutils`, uses the implicit user identity via `DefaultAzureCredential`, and reads workspace context from `notebookutils.runtime.context`.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table

FABRIC_API   = "https://api.fabric.microsoft.com/v1"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

# --- Detect runtime: Fabric notebook vs local .venv -------------------------
try:
    import notebookutils  # only present inside Fabric runtime
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if IN_FABRIC:
    from azure.identity import DefaultAzureCredential
    ctx        = notebookutils.runtime.context
    WORKSPACE  = ctx.get("currentWorkspaceName")
    TENANT_ID  = ctx.get("tenantId", "")
    KQLDB_NAME = os.environ.get("FABRIC_KQLDB_NAME", "kql_telemetry")
    cred       = DefaultAzureCredential()
    print(f"[auth] Fabric runtime (workspace={WORKSPACE})")
else:
    from dotenv import load_dotenv
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / ".env").exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent
    load_dotenv(REPO_ROOT / ".env")
    sys.path.insert(0, str(REPO_ROOT / "tools"))
    from _fabric_auth import get_credential  # noqa: E402
    TENANT_ID  = os.environ["FABRIC_TENANT_ID"]
    WORKSPACE  = os.environ["FABRIC_WORKSPACE_NAME"]
    KQLDB_NAME = os.environ["FABRIC_KQLDB_NAME"]
    cred       = get_credential(TENANT_ID, FABRIC_SCOPE, REPO_ROOT)
    print(f"[auth] local .venv (.env @ {REPO_ROOT})")

fabric_token = cred.get_token(FABRIC_SCOPE).token
headers      = {"Authorization": f"Bearer {fabric_token}"}

# Resolve workspace + KQL DB ids -> queryServiceUri.
ws = next(w for w in requests.get(f"{FABRIC_API}/workspaces", headers=headers).json()["value"]
          if w["displayName"] == WORKSPACE)
ws_id = ws["id"]
db    = next(d for d in requests.get(f"{FABRIC_API}/workspaces/{ws_id}/kqlDatabases", headers=headers).json()["value"]
             if d["displayName"] == KQLDB_NAME)
query_uri = db["properties"]["queryServiceUri"]
print("queryServiceUri:", query_uri)

kcsb  = KustoConnectionStringBuilder.with_azure_token_credential(query_uri, cred)
kusto = KustoClient(kcsb)

[auth] using cached credentials
queryServiceUri: https://trd-xr97y56tuzzkxy5cgp.z5.kusto.fabric.microsoft.com


## 3. Pull data and build windows

In [6]:
q = f"""
raw_telemetry
| where machine_id == '{MACHINE}' and sensor_id == '{SENSOR}'
| where ts > now() - {LOOKBACK}
| order by ts asc
| project ts, value
"""
res = kusto.execute(KQLDB_NAME, q)
df  = dataframe_from_result_table(res.primary_results[0])
print("rows:", len(df))
df.head()

rows: 464


,ts,value
0,2026-05-14 09:32:02.056479+00:00,61.7575
1,2026-05-14 09:32:03.056563+00:00,61.7154
2,2026-05-14 09:32:04.056562+00:00,61.4965
3,2026-05-14 09:32:05.056561+00:00,61.1531
4,2026-05-14 09:32:06.056582+00:00,61.132


In [ ]:
values = df["value"].to_numpy(dtype=np.float32)
n_full = (len(values) // WINDOW_SIZE) * WINDOW_SIZE
X = values[:n_full].reshape(-1, WINDOW_SIZE, 1)
if X.shape[0] < 4:
    raise RuntimeError(f"Need more windows to train ({X.shape[0]} available). Increase LOOKBACK or let the simulator run longer.")
print("training tensor:", X.shape)

training tensor: (7, 64, 1)


## 4. Model + training

In [8]:
import torch
from torch import nn

torch.manual_seed(0)

class WindowAE(nn.Module):
    def __init__(self, n_features: int, hidden: int = 32):
        super().__init__()
        self.enc = nn.LSTM(n_features, hidden, batch_first=True)
        self.dec = nn.LSTM(hidden, n_features, batch_first=True)
    def forward(self, x):
        h, _   = self.enc(x)
        out, _ = self.dec(h)
        return out

model = WindowAE(n_features=1)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
loss  = nn.MSELoss()

t = torch.from_numpy(X)
for epoch in range(EPOCHS):
    opt.zero_grad()
    e = loss(model(t), t)
    e.backward()
    opt.step()
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:>3}  loss {e.item():.6f}")

final_loss = float(e.item())

epoch   0  loss 3694.582764
epoch   5  loss 3686.770020
epoch  10  loss 3684.755127
epoch  15  loss 3683.718750
epoch  20  loss 3682.966797
epoch  25  loss 3682.375488
epoch  29  loss 3681.980957


## 5. Export to ONNX (with `ScoreWrapper`)

The wrapper returns **a single scalar score per window** (reconstruction MSE) — exactly what `score_univariate_onnx` expects in KQL.

In [17]:
class ScoreWrapper(nn.Module):
    def __init__(self, ae: nn.Module):
        super().__init__()
        self.ae = ae
    def forward(self, x):
        recon = self.ae(x)
        return ((recon - x) ** 2).mean(dim=(1, 2))

wrapped = ScoreWrapper(model).eval()
dummy   = torch.zeros((1, WINDOW_SIZE, 1), dtype=torch.float32)

buf = io.BytesIO()
torch.onnx.export(
    wrapped, dummy, buf,
    input_names=["window"], output_names=["score"],
    dynamic_axes={"window": {0: "batch"}, "score": {0: "batch"}},
    opset_version=17,
)
onnx_bytes = buf.getvalue()

# Sandbox onnxruntime supports IR <= 9. Force-downgrade if needed.
import onnx
mp = onnx.load_from_string(onnx_bytes)
if mp.ir_version > 9:
    mp.ir_version = 9
    onnx_bytes = mp.SerializeToString()

onnx_b64   = base64.b64encode(onnx_bytes).decode("ascii")
print(f"ONNX size: {len(onnx_bytes)/1024:.1f} KB  (base64 {len(onnx_b64)/1024:.1f} KB), ir={mp.ir_version}")

C:\Users\faustopalma\AppData\Local\Temp\ipykernel_34312\2033215096.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0514 11:43:55.329000 34312 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0514 11:43:57.080000 34312 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0514 11:43:57.

[torch.onnx] Obtain model graph for `ScoreWrapper([...]` with `torch.export.export(..., strict=False)`...


C:\Users\faustopalma\AppData\Local\Programs\Python\Python313\Lib\contextlib.py:148: UserWarning: The tensor attributes self.ae.enc._flat_weights[0], self.ae.enc._flat_weights[1], self.ae.enc._flat_weights[2], self.ae.enc._flat_weights[3], self.ae.dec._flat_weights[0], self.ae.dec._flat_weights[1], self.ae.dec._flat_weights[2], self.ae.dec._flat_weights[3] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `ScoreWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\faustopalma\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\faustopalma\OneDrive - Microsoft\Documents\Customers\Other\Iveco\anomalydetection\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_ver

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX size: 57.5 KB  (base64 76.7 KB), ir=9


In [ ]:
# Sanity check: load the ONNX in onnxruntime locally and verify input/output shapes.
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_bytes)
    out = sess.run(None, {"window": X[:3]})[0]
    print("local onnxruntime score sample:", out)
except ImportError:
    print("(onnxruntime not installed locally, skipping sanity check — does not block upload)")

local onnxruntime score sample: [3771.8872 3840.7275 3586.6594]


## 6. Upload into the `models` table

We use `.set-or-append models <| print ...`: a command-mode admin call, no separate ingestion endpoint (Fabric Eventhouse does not expose one like ADX does).

In [ ]:
next_v = 1
try:
    res = kusto.execute(KQLDB_NAME, f"models | where name == '{MODEL_NAME}' | summarize v = max(version)")
    cur = dataframe_from_result_table(res.primary_results[0])["v"].iloc[0]
    if cur is not None and not pd.isna(cur):
        next_v = int(cur) + 1
except Exception as ex:
    print("warn: lookup version failed, defaulting to 1:", ex)

created_at = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
metadata   = json.dumps({
    "final_loss":  final_loss,
    "n_windows":   int(X.shape[0]),
    "trained_at":  created_at,
    "machine":     MACHINE,
    "sensor":      SENSOR,
})

# Avoid escaping issues with base64 (which has no quotes anyway):
# build the command as a single string.
cmd = (
    ".set-or-append models <|\n"
    "print "
    f"name='{MODEL_NAME}', "
    f"version=int({next_v}), "
    f"created_at=datetime({created_at}), "
    "framework='onnx', "
    f"window_size=int({WINDOW_SIZE}), "
    "sensors=dynamic(null), "
    f"payload='{onnx_b64}', "
    f"metadata=todynamic('{metadata}')"
)
kusto.execute_mgmt(KQLDB_NAME, cmd)
print(f"uploaded {MODEL_NAME} v{next_v}")

uploaded univariate_ae__temperature_motor v2


## 7. Smoke test in KQL

In [19]:
smoke = f"""
score_univariate_onnx(
    '{MODEL_NAME}', '{MACHINE}', '{SENSOR}',
    {WINDOW_SIZE}, {LOOKBACK}, 0.0
)
| top 5 by window_end desc
| project window_start, window_end, score, model_version
"""
res = kusto.execute(KQLDB_NAME, smoke)
dataframe_from_result_table(res.primary_results[0])

,window_start,window_end,score,model_version
0,2026-05-14 09:42:42.056650+00:00,2026-05-14 09:43:45.056544+00:00,3861.207031,2
1,2026-05-14 09:41:38.056683+00:00,2026-05-14 09:42:41.056585+00:00,3665.420898,2
2,2026-05-14 09:40:34.056549+00:00,2026-05-14 09:41:37.056565+00:00,3428.57959,2
3,2026-05-14 09:39:30.056568+00:00,2026-05-14 09:40:33.056575+00:00,3502.133057,2
4,2026-05-14 09:38:26.056570+00:00,2026-05-14 09:39:29.056588+00:00,3763.911621,2
